In [1]:
# BOOK RECOMMENDATION SYSTEM 

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer   # ← UPGRADED from CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.porter import PorterStemmer
import nltk
import pickle

nltk.download('punkt')

# 1. LOAD DATA 
df = pd.read_csv('books.csv', on_bad_lines='skip')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\PMLS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Shape: (11123, 12)
Columns: ['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13', 'language_code', '  num_pages', 'ratings_count', 'text_reviews_count', 'publication_date', 'publisher']


,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic


In [3]:
# 2. EXPLORE DATA ───────────────────────────────────────────────────────────
print(df.info())
print("\nMissing values:\n", df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11123 entries, 0 to 11122
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              11123 non-null  int64  
 1   title               11123 non-null  object 
 2   authors             11123 non-null  object 
 3   average_rating      11123 non-null  float64
 4   isbn                11123 non-null  object 
 5   isbn13              11123 non-null  int64  
 6   language_code       11123 non-null  object 
 7     num_pages         11123 non-null  int64  
 8   ratings_count       11123 non-null  int64  
 9   text_reviews_count  11123 non-null  int64  
 10  publication_date    11123 non-null  object 
 11  publisher           11123 non-null  object 
dtypes: float64(1), int64(5), object(6)
memory usage: 1.0+ MB
None

Missing values:
 bookID                0
title                 0
authors               0
average_rating        0
isbn                  0


In [5]:
# 3. KEEP USEFUL COLUMNS

available_cols = df.columns.tolist()
keep = ['bookID', 'title', 'authors', 'average_rating', 'language_code']

for optional in ['genres', 'description', 'publisher']:
    if optional in available_cols:
        keep.append(optional)
        print(f"✅ Found optional column: '{optional}' — will use it in tags")
    else:
        print(f"ℹ️  Column '{optional}' not in dataset — skipping")

df = df[keep].copy()
df.dropna(subset=['title', 'authors'], inplace=True)
df.drop_duplicates(subset='title', inplace=True)
df.reset_index(drop=True, inplace=True)

print("\nAfter cleaning:", df.shape)
df.head()

ℹ️  Column 'genres' not in dataset — skipping
ℹ️  Column 'description' not in dataset — skipping
✅ Found optional column: 'publisher' — will use it in tags

After cleaning: (10348, 6)


,bookID,title,authors,average_rating,language_code,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,eng,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,eng,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,eng,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,eng,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,eng,Scholastic


In [7]:
# 4. BUILD TAGS 

def clean_name(name: str) -> str:
    """Remove spaces so multi-word names become one token."""
    return str(name).replace(" ", "").replace("-", "").lower()

def build_tags(row) -> str:
    parts = []

    # Title words
    parts.extend(str(row['title']).lower().split())

    # Author(s) collapsed
    for author in str(row['authors']).split('/'):
        parts.append(clean_name(author.strip()))

    # Language
    if 'language_code' in row and pd.notna(row['language_code']):
        parts.append(str(row['language_code']).lower().strip())

    # Genres if available
    if 'genres' in row and pd.notna(row['genres']):
        genre_text = str(row['genres']).lower()
        for sep in ['|', ',', ';']:
            genre_text = genre_text.replace(sep, ' ')
        genre_text = genre_text.replace('[', '').replace(']', '').replace("'", '')
        parts.extend(genre_text.split())

    # Description if available 
    if 'description' in row and pd.notna(row['description']):
        desc_words = str(row['description']).lower().split()
        parts.extend(desc_words[:80])   

    # Publisher if available
    if 'publisher' in row and pd.notna(row['publisher']):
        parts.append(clean_name(str(row['publisher'])))

    return " ".join(parts)

df['tags'] = df.apply(build_tags, axis=1)

print("\nSample tags (first 3 books):")
print(df[['title', 'tags']].head(3).to_string())


Sample tags (first 3 books):
                                                          title                                                                                                     tags
0     Harry Potter and the Half-Blood Prince (Harry Potter  #6)     harry potter and the half-blood prince (harry potter #6) j.k.rowling marygrandpré eng scholasticinc.
1  Harry Potter and the Order of the Phoenix (Harry Potter  #5)  harry potter and the order of the phoenix (harry potter #5) j.k.rowling marygrandpré eng scholasticinc.
2    Harry Potter and the Chamber of Secrets (Harry Potter  #2)                     harry potter and the chamber of secrets (harry potter #2) j.k.rowling eng scholastic


In [15]:
# 5. STEMMING

ps = PorterStemmer()

def stem(text: str) -> str:
    return " ".join(ps.stem(w) for w in text.split())

df['tags'] = df['tags'].apply(stem)

print("\nExample tags after stemming:")
print(df['tags'][0][:300])


Example tags after stemming:
harri potter and the half-blood princ (harri potter #6) j.k.rowl marygrandpré eng scholasticinc.


In [17]:
# 6. TF-IDF VECTORIZATION 

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2)   
)
vectors = tfidf.fit_transform(df['tags']).toarray()

print("\nTF-IDF vectors shape:", vectors.shape)


TF-IDF vectors shape: (10348, 5000)


In [19]:
# 7. COSINE SIMILARITY 
similarity = cosine_similarity(vectors)
print("Similarity matrix shape:", similarity.shape)

Similarity matrix shape: (10348, 10348)


In [23]:
# 8. RECOMMENDATION FUNCTION 
def recommend(book_title: str, n: int = 5):
    matches = df[df['title'].str.lower() == book_title.lower()]

    if matches.empty:
        # Partial match fallback
        partial = df[df['title'].str.lower().str.contains(
            book_title.lower(), na=False, regex=False
        )]
        if partial.empty:
            print(f"❌ Book '{book_title}' not found.")
            print("   Try: search_book('keyword')")
            return
        print(f"⚠️  Exact match not found. Closest match:")
        book_title = partial.iloc[0]['title']
        print(f"   Using: '{book_title}'\n")

    index = df[df['title'].str.lower() == book_title.lower()].index[0]
    distances = sorted(
        list(enumerate(similarity[index])),
        reverse=True,
        key=lambda x: x[1]
    )[1: n + 1]

    print(f"📚 Books similar to '{book_title}':\n")
    for rank, (i, score) in enumerate(distances, 1):
        row    = df.iloc[i]
        rating = row.get('average_rating', 'N/A')
        print(f"  {rank}. {row['title']}")
        print(f"     Author : {row['authors']}")
        print(f"     Rating : {rating}  |  Similarity : {score:.4f}")
        print()


def search_book(keyword: str, limit: int = 10):
    """Search books by partial title keyword."""
    results = df[df['title'].str.lower().str.contains(
        keyword.lower(), na=False, regex=False
    )]['title'].tolist()
    if results:
        print(f"🔍 Books matching '{keyword}':")
        for r in results[:limit]:
            print("  -", r)
    else:
        print("No books found.")

In [25]:
# 9. TEST 
recommend('Harry Potter and the Chamber of Secrets')
recommend('The Hobbit')
recommend('twilight')      
search_book('Lord')

⚠️  Exact match not found. Closest match:
   Using: 'Harry Potter and the Chamber of Secrets (Harry Potter  #2)'

📚 Books similar to 'Harry Potter and the Chamber of Secrets (Harry Potter  #2)':

  1. Harry Potter Collection (Harry Potter  #1-6)
     Author : J.K. Rowling
     Rating : 4.73  |  Similarity : 0.9606

  2. Harry Potter ve Sırlar Odası (Harry Potter  #2)
     Author : J.K. Rowling/Sevin Okyay
     Rating : 4.42  |  Similarity : 0.8778

  3. Harry Potter and the Goblet of Fire (Harry Potter  #4)
     Author : J.K. Rowling
     Rating : 4.56  |  Similarity : 0.8552

  4. Harry Potter and the Philosopher's Stone (Harry Potter  #1)
     Author : J.K. Rowling
     Rating : 4.47  |  Similarity : 0.8474

  5. Harry Potter Y La Piedra Filosofal (Harry Potter  #1)
     Author : J.K. Rowling
     Rating : 4.47  |  Similarity : 0.8138

📚 Books similar to 'The Hobbit':

  1. The Hobbit: Or There and Back Again
     Author : J.R.R. Tolkien
     Rating : 4.27  |  Similarity : 0.8840

  

In [27]:
# 10. SAVE MODEL FILES 
pickle.dump(df,         open('book_list.pkl',   'wb'))
pickle.dump(similarity, open('similarity.pkl',  'wb'))

print("✅ Model files saved:")
print("   book_list.pkl")
print("   similarity.pkl")


✅ Model files saved:
   book_list.pkl
   similarity.pkl
